# TGNMemory Unsupervised Anomaly Detection

This notebook keeps the same preprocessing pipeline and TGNMemory backbone, but changes the learning objective to unsupervised edge-feature reconstruction.

Training does not use `Is Laundering` labels. Labels are only used for validation/test diagnostics (ROC-AUC, PR-AUC, thresholded report).

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from time import perf_counter
from typing import Literal
import random

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from torch.nn import Linear
from torch.optim import AdamW
from torch_geometric.data import TemporalData
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.nn import TGNMemory, TransformerConv
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastNeighborLoader,
    MeanAggregator,
)
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_path = Path('../dataset')
artifact_dir = Path('./preprocessors')
checkpoint_dir = Path('./checkpoints/tgn_unsupervised')
log_dir = Path('./logs/tensorboard')

artifact_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

accounts_path = dataset_path / 'HI-Small_accounts.csv'
transactions_path = dataset_path / 'HI-Small_Trans.csv'
device = torch.device('cpu')

print(f'{accounts_path=}')
print(f'{transactions_path=}')
print(f'{device=}')

accounts_path=PosixPath('../dataset/HI-Small_accounts.csv')
transactions_path=PosixPath('../dataset/HI-Small_Trans.csv')
device=device(type='cpu')


In [3]:
accounts = pd.read_csv(accounts_path)
accounts['Bank Name'] = accounts['Bank Name'].str.split(' #').str[0]
accounts['Entity Name'] = accounts['Entity Name'].str.split(' #').str[0]
print(f'{accounts.shape=}')
accounts.head()

accounts.shape=(518581, 5)


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank,331579,80B779D80,80062E240,Sole Proprietorship
1,Canada Bank,210,809D86900,800C998A0,Corporation
2,UK Bank,21884,80812BE00,800C47F50,Partnership
3,Germany Bank,32742,81047F300,80096F0B0,Corporation
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation


In [4]:
transactions = pd.read_csv(transactions_path)
transactions = transactions.rename(columns={'Account': 'From Account', 'Account.1': 'To Account'})
transactions['Timestamp'] = pd.to_datetime(transactions['Timestamp'])
print(f'{transactions.shape=}')
transactions.head()

transactions.shape=(5078345, 11)


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## Preprocessing

The preprocessing is unchanged from your supervised notebook.

In [5]:
accounts_iterator = accounts.loc[:, ['Bank ID', 'Account Number']].itertuples()
account_to_idx = {(bank_id, account_number): idx for idx, (_, bank_id, account_number) in enumerate(accounts_iterator)}

vertex_transformer = ColumnTransformer([
    ('categorical_features', OneHotEncoder(sparse_output=False, drop='first'), ['Entity Name']),
], remainder='drop')

vertices = torch.tensor(vertex_transformer.fit_transform(accounts), dtype=torch.float32)
joblib.dump(vertex_transformer, artifact_dir / 'vertex_preprocessor.joblib')
print(f'{vertices.shape=}')

vertices.shape=torch.Size([518581, 5])


In [6]:
source_keys = list(zip(transactions['From Bank'], transactions['From Account']))
destination_keys = list(zip(transactions['To Bank'], transactions['To Account']))
transactions['source'] = pd.Series(source_keys).map(account_to_idx)
transactions['destination'] = pd.Series(destination_keys).map(account_to_idx)
edge_index = torch.tensor(transactions[['source', 'destination']].values.T, dtype=torch.long)
print(f'{edge_index.shape=}')

edge_index.shape=torch.Size([2, 5078345])


In [7]:
categorical_features = ['Payment Format', 'Receiving Currency', 'Payment Currency']
numerical_features = ['Amount Received', 'Amount Paid']

edge_transformer = ColumnTransformer([
    ('categorical', OneHotEncoder(sparse_output=False), categorical_features),
    ('numerical', RobustScaler(), numerical_features),
], remainder='passthrough', verbose_feature_names_out=False)
edge_transformer.set_output(transform='pandas')

preprocessed_transactions = edge_transformer.fit_transform(transactions)
to_remove = ['From Bank', 'From Account', 'To Account', 'To Bank', 'Is Laundering', 'Timestamp']
preprocessed_transactions = preprocessed_transactions.drop(columns=to_remove)
edge_features = torch.tensor(preprocessed_transactions.values, dtype=torch.float32)
joblib.dump(edge_transformer, artifact_dir / 'edge_preprocessor.joblib')
print(f'{edge_features.shape=}')
preprocessed_transactions.head()

edge_features.shape=torch.Size([5078345, 41])


,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,...,Payment Currency_Shekel,Payment Currency_Swiss Franc,Payment Currency_UK Pound,Payment Currency_US Dollar,Payment Currency_Yen,Payment Currency_Yuan,Amount Received,Amount Paid,source,destination
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.187976,0.188453,435512,435512
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,-0.116009,-0.116774,65242,474699
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.090575,1.094744,65597,65597
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.114772,0.114950,475408,475408
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,2.899963,2.911532,475619,475619


In [8]:
edge_labels = torch.tensor(transactions['Is Laundering'].values, dtype=torch.float32)
unix_timestamp = transactions['Timestamp'].astype('int64') // 10**9

data = TemporalData(
    src=edge_index[0],
    dst=edge_index[1],
    t=torch.tensor(unix_timestamp.values, dtype=torch.long),
    msg=edge_features,
    y=edge_labels,
).to(device)

train_data, val_data, test_data = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
train_loader = TemporalDataLoader(train_data, batch_size=512)
val_loader = TemporalDataLoader(val_data, batch_size=512)
test_loader = TemporalDataLoader(test_data, batch_size=512)
neighbor_loader = LastNeighborLoader(data.num_nodes, size=15, device=device)

print(f'{train_data.num_events=}, {val_data.num_events=}, {test_data.num_events=}')

train_data.num_events=3557613, val_data.num_events=763462, test_data.num_events=757270


## Model

Backbone (`TGNMemory` + temporal GNN) is unchanged. Only the final head and loss are changed:
- Head predicts edge features (reconstruction target)
- Loss is unsupervised reconstruction error (MSE)
- Anomaly score is per-edge reconstruction error

In [9]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    torch.use_deterministic_algorithms(True)


def threshold_from_percentile(scores: np.ndarray, percentile: float = 99.0) -> float:
    if scores.size == 0:
        return 0.0
    return float(np.percentile(scores, percentile))


def compute_threshold_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float) -> dict[str, float]:
    y_pred = (y_score >= threshold).astype(np.int64)
    metrics = {
        'threshold': float(threshold),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }
    if np.unique(y_true).size > 1:
        metrics['roc_auc'] = float(roc_auc_score(y_true, y_score))
        metrics['pr_auc'] = float(average_precision_score(y_true, y_score))
    else:
        metrics['roc_auc'] = float('nan')
        metrics['pr_auc'] = float('nan')
    return metrics

In [10]:
class GraphAttentionEmbedding(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, msg_dim: int, time_enc) -> None:
        super().__init__()
        self.time_enc = time_enc
        edge_dim = msg_dim + time_enc.out_channels
        self.conv = TransformerConv(
            in_channels,
            out_channels // 2,
            heads=2,
            dropout=0.1,
            edge_dim=edge_dim,
        )

    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = last_update[edge_index[0]] - t
        rel_t_enc = self.time_enc(rel_t.to(x.dtype))
        edge_attr = torch.cat([rel_t_enc, msg], dim=-1)
        return self.conv(x, edge_index, edge_attr)


class EdgeReconstructionHead(torch.nn.Module):
    def __init__(self, in_channels: int, edge_dim: int) -> None:
        super().__init__()
        self.lin = Linear(in_channels * 2, in_channels)
        self.lin_out = Linear(in_channels, edge_dim)

    def forward(self, z_src, z_dst):
        h = torch.cat([z_src, z_dst], dim=-1)
        h = self.lin(h).relu()
        return self.lin_out(h)

In [11]:
class TemporalUnsupervisedTrainer:
    def __init__(
        self,
        memory: TGNMemory,
        gnn: torch.nn.Module,
        edge_head: torch.nn.Module,
        data: TemporalData,
        neighbor_loader: LastNeighborLoader,
        train_loader: TemporalDataLoader,
        val_loader: TemporalDataLoader,
        test_loader: TemporalDataLoader,
        optimizer_cls: type[torch.optim.Optimizer],
        checkpoint_dir: Path,
        log_dir: Path,
        device: Literal['cpu', 'mps'] = 'cpu',
    ) -> None:
        self.device = device
        self.memory = memory.to(self.device)
        self.gnn = gnn.to(self.device)
        self.edge_head = edge_head.to(self.device)
        self.data = data
        self.neighbor_loader = neighbor_loader
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.optimizer = optimizer_cls(
            list(self.memory.parameters()) + list(self.gnn.parameters()) + list(self.edge_head.parameters()),
            lr=1e-4,
        )
        self.checkpoint_dir = checkpoint_dir
        self.log_dir = log_dir
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.assoc = torch.empty(data.num_nodes, dtype=torch.long, device=self.device)
        self.epochs_run = 0
        self.best_val_loss = float('inf')
        self.best_threshold = 0.0
        self.history: list[dict[str, float]] = []

    def train_mode(self) -> None:
        self.memory.train()
        self.gnn.train()
        self.edge_head.train()

    def eval_mode(self) -> None:
        self.memory.eval()
        self.gnn.eval()
        self.edge_head.eval()

    def _reset_temporal_state(self) -> None:
        self.memory.reset_state()
        self.neighbor_loader.reset_state()

    def _forward_batch(self, batch):
        n_id, edge_index, e_id = self.neighbor_loader(batch.n_id)
        self.assoc[n_id] = torch.arange(n_id.size(0), device=self.device)

        z, last_update = self.memory(n_id)
        z = self.gnn(
            z,
            last_update,
            edge_index,
            self.data.t[e_id].to(self.device),
            self.data.msg[e_id].to(self.device),
        )

        pred_msg = self.edge_head(
            z[self.assoc[batch.src]],
            z[self.assoc[batch.dst]],
        )
        # Per-edge MSE score is used as anomaly score.
        per_edge_score = ((pred_msg - batch.msg) ** 2).mean(dim=-1)
        loss = per_edge_score.mean()
        return loss, per_edge_score

    def _run_train_batch(self, batch) -> float:
        batch = batch.to(self.device)
        self.optimizer.zero_grad()

        loss, _ = self._forward_batch(batch)
        loss.backward()
        self.optimizer.step()

        self.memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        self.neighbor_loader.insert(batch.src, batch.dst)
        self.memory.detach()
        return float(loss.item())

    def _train_epoch(self) -> float:
        losses = []
        progress_bar = tqdm(self.train_loader, total=len(self.train_loader), leave=False)
        for batch in progress_bar:
            batch_loss = self._run_train_batch(batch)
            losses.append(batch_loss)
            progress_bar.set_postfix({'loss': f'{np.mean(losses):.4f}'})
        return float(np.mean(losses)) if losses else float('nan')

    @torch.no_grad()
    def _predict_loader(self, loader: TemporalDataLoader) -> tuple[float, np.ndarray, np.ndarray]:
        self.eval_mode()
        losses = []
        y_true_batches = []
        score_batches = []

        for batch in loader:
            batch = batch.to(self.device)
            loss, scores = self._forward_batch(batch)
            losses.append(float(loss.item()))
            y_true_batches.append(batch.y.detach().cpu().to(torch.int64))
            score_batches.append(scores.detach().cpu())

            self.memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
            self.neighbor_loader.insert(batch.src, batch.dst)

        y_true = torch.cat(y_true_batches).numpy() if y_true_batches else np.array([], dtype=np.int64)
        y_score = torch.cat(score_batches).numpy() if score_batches else np.array([], dtype=np.float32)
        avg_loss = float(np.mean(losses)) if losses else float('nan')
        return avg_loss, y_true, y_score

    def _evaluate_split(self, loader: TemporalDataLoader, threshold: float | None = None) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
        self._reset_temporal_state()
        avg_loss, y_true, y_score = self._predict_loader(loader)

        if threshold is None:
            calibrated_threshold = threshold_from_percentile(y_score, percentile=99.0)
        else:
            calibrated_threshold = float(threshold)

        metrics = compute_threshold_metrics(y_true, y_score, calibrated_threshold)
        metrics['loss'] = float(avg_loss)
        return avg_loss, metrics, y_true, y_score

    def evaluate_loader(self, loader: TemporalDataLoader, threshold: float | None = None) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
        return self._evaluate_split(loader, threshold=threshold)

    def evaluate_test(self, threshold: float | None = None) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
        return self._evaluate_split(self.test_loader, threshold=threshold)

    def _save_checkpoint(self, path: Path, epoch: int, metrics: dict[str, float]) -> None:
        checkpoint = {
            'epoch': epoch,
            'best_val_loss': self.best_val_loss,
            'best_threshold': self.best_threshold,
            'memory_state_dict': self.memory.state_dict(),
            'gnn_state_dict': self.gnn.state_dict(),
            'edge_head_state_dict': self.edge_head.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metrics': metrics,
        }
        torch.save(checkpoint, path)

    def load_checkpoint(self, path: Path) -> dict:
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.memory.load_state_dict(checkpoint['memory_state_dict'])
        self.gnn.load_state_dict(checkpoint['gnn_state_dict'])
        self.edge_head.load_state_dict(checkpoint['edge_head_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.best_val_loss = float(checkpoint.get('best_val_loss', float('inf')))
        self.best_threshold = float(checkpoint.get('best_threshold', 0.0))
        self.epochs_run = int(checkpoint.get('epoch', 0)) + 1
        return checkpoint

    def fit(self, max_epochs: int) -> list[dict[str, float]]:
        run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
        writer_path = self.log_dir / run_id
        history: list[dict[str, float]] = []

        with SummaryWriter(str(writer_path)) as writer:
            for epoch in range(self.epochs_run, max_epochs):
                self._reset_temporal_state()
                self.train_mode()
                start_time = perf_counter()
                train_loss = self._train_epoch()
                train_time = perf_counter() - start_time

                self.eval_mode()
                val_loss, val_metrics, _, _ = self._evaluate_split(self.val_loader)
                self.best_threshold = val_metrics['threshold']

                epoch_summary = {
                    'epoch': float(epoch),
                    'train_loss': float(train_loss),
                    'val_loss': float(val_loss),
                    'val_pr_auc': float(val_metrics['pr_auc']),
                    'val_roc_auc': float(val_metrics['roc_auc']),
                    'val_f1': float(val_metrics['f1']),
                    'val_precision': float(val_metrics['precision']),
                    'val_recall': float(val_metrics['recall']),
                    'threshold': float(val_metrics['threshold']),
                    'train_time_seconds': float(train_time),
                }
                history.append(epoch_summary)
                self.history.append(epoch_summary)

                writer.add_scalar('Loss/train', train_loss, epoch)
                writer.add_scalar('Loss/validation', val_loss, epoch)
                writer.add_scalar('Metrics/val_pr_auc', val_metrics['pr_auc'], epoch)
                writer.add_scalar('Metrics/val_roc_auc', val_metrics['roc_auc'], epoch)
                writer.add_scalar('Metrics/val_f1', val_metrics['f1'], epoch)
                writer.add_scalar('Threshold/validation', val_metrics['threshold'], epoch)

                latest_path = self.checkpoint_dir / 'latest.pt'
                self._save_checkpoint(latest_path, epoch, val_metrics)

                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    best_path = self.checkpoint_dir / 'best.pt'
                    self._save_checkpoint(best_path, epoch, val_metrics)

                print(f'Epoch: {epoch} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Time: {train_time:.2f}s')
                print(f"Val threshold: {val_metrics['threshold']:.6f} | PR AUC: {val_metrics['pr_auc']:.6f} | ROC AUC: {val_metrics['roc_auc']:.6f}")

                self.epochs_run += 1

        return history

In [12]:
set_seed(124)

embedding_dim = memory_dim = time_dim = 64
memory = TGNMemory(
    data.num_nodes,
    data.msg.size(-1),
    memory_dim,
    time_dim,
    message_module=IdentityMessage(data.msg.size(-1), memory_dim, time_dim),
    aggregator_module=MeanAggregator(),
)

gnn = GraphAttentionEmbedding(
    in_channels=memory_dim,
    out_channels=embedding_dim,
    msg_dim=data.msg.size(-1),
    time_enc=memory.time_enc,
)

edge_head = EdgeReconstructionHead(in_channels=embedding_dim, edge_dim=data.msg.size(-1))

trainer = TemporalUnsupervisedTrainer(
    memory=memory,
    gnn=gnn,
    edge_head=edge_head,
    data=data,
    neighbor_loader=neighbor_loader,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer_cls=AdamW,
    checkpoint_dir=checkpoint_dir,
    log_dir=log_dir,
    device=device,
)

history = trainer.fit(max_epochs=5)
history[-1] if history else {}

/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


Epoch: 0 | Train Loss: 1160721306.293280 | Val Loss: 2736487965.254692 | Time: 168.98s
Val threshold: 9916791808.000000 | PR AUC: 0.000914 | ROC AUC: 0.482811


Epoch: 1 | Train Loss: 1076897781.858685 | Val Loss: 2752045956.117962 | Time: 172.00s
Val threshold: 9925275648.000000 | PR AUC: 0.000904 | ROC AUC: 0.480631


Epoch: 2 | Train Loss: 1046678109.464383 | Val Loss: 2742965534.713137 | Time: 165.37s
Val threshold: 9929271296.000000 | PR AUC: 0.000898 | ROC AUC: 0.478276


Epoch: 3 | Train Loss: 1036061690.469420 | Val Loss: 2733529895.892761 | Time: 159.53s
Val threshold: 9924500480.000000 | PR AUC: 0.000894 | ROC AUC: 0.475815


Epoch: 4 | Train Loss: 1028420627.949345 | Val Loss: 2723189167.442359 | Time: 169.88s
Val threshold: 9916341248.000000 | PR AUC: 0.000893 | ROC AUC: 0.475691


{'epoch': 4.0,
 'train_loss': 1028420627.9493452,
 'val_loss': 2723189167.4423594,
 'val_pr_auc': 0.0008931174797090768,
 'val_roc_auc': 0.47569095734178546,
 'val_f1': 0.0016768475266498981,
 'val_precision': 0.00091683038637852,
 'val_recall': 0.00980392156862745,
 'threshold': 9916341248.0,
 'train_time_seconds': 169.87925645800715}

In [13]:
best_checkpoint = checkpoint_dir / 'best.pt'
trainer.load_checkpoint(best_checkpoint)

val_loss, val_metrics, val_true, val_score = trainer.evaluate_loader(val_loader, threshold=trainer.best_threshold)
print(f'{val_loss=:.6f}')
print(val_metrics)
print(classification_report(val_true, (val_score >= trainer.best_threshold).astype(int), digits=5, zero_division=0))

test_loss, test_metrics, test_true, test_score = trainer.evaluate_test(threshold=trainer.best_threshold)
print(f'{test_loss=:.6f}')
print(test_metrics)
print(classification_report(test_true, (test_score >= trainer.best_threshold).astype(int), digits=5, zero_division=0))

val_loss=2723189167.442359
{'threshold': 9916341248.0, 'precision': 0.00091683038637852, 'recall': 0.00980392156862745, 'f1': 0.0016768475266498981, 'roc_auc': 0.47569095734178546, 'pr_auc': 0.0008931174797090768, 'loss': 2723189167.4423594}
              precision    recall  f1-score   support

           0    0.99906   0.99000   0.99451    762748
           1    0.00092   0.00980   0.00168       714

    accuracy                        0.98908    763462
   macro avg    0.49999   0.49990   0.49809    763462
weighted avg    0.99813   0.98908   0.99358    763462

test_loss=3025946130.421622
{'threshold': 9916341248.0, 'precision': 0.0003480682213713888, 'recall': 0.0029469548133595285, 'f1': 0.000622600394313583, 'roc_auc': 0.4769765898139773, 'pr_auc': 0.0012026790720567676, 'loss': 3025946130.421622}
              precision    recall  f1-score   support

           0    0.99864   0.98861   0.99360    756252
           1    0.00035   0.00295   0.00062      1018

    accuracy           